# URA Hackathon 2026 - Pipeline v4.0 (Merged)

**v4.0:** PaddleOCR CPU (TANAHIVER6) + anti-hallucination token filter + post_process

**Merge:** TANAHIVER6 OCR/ML + our brand rules/post_process

**Pipeline:** BRAND_RULES -> _assign_readable_tokens -> ProductPredictor(LogReg) -> post_process


In [1]:
import os, sys, subprocess
from concurrent.futures import ThreadPoolExecutor

CPU_THREADS = max(1, os.cpu_count() or 2)
PIP_WORKERS = min(CPU_THREADS, 4)
print(f'CPU threads: {CPU_THREADS} | pip workers: {PIP_WORKERS}')

for _env in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_env, str(CPU_THREADS))

def _pip(*pkgs):
    subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=True)

print('Installing: paddlepaddle (CPU) + tqdm + scikit-learn ...')
with ThreadPoolExecutor(max_workers=PIP_WORKERS) as pool:
    jobs = [
        pool.submit(_pip, 'paddlepaddle'),
        pool.submit(_pip, 'tqdm'),
        pool.submit(_pip, 'scikit-learn'),
    ]
    for j in jobs: j.result()
_pip('paddleocr>=2.7,<3')
print('Install done (CPU-only PaddleOCR)')


CPU threads: 4 | pip workers: 4
Installing: paddlepaddle (CPU) + tqdm + scikit-learn ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.6 MB/s eta 0:00:00
Install done (CPU-only PaddleOCR)


In [2]:
import os, re, csv, time, warnings, unicodedata
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image, ImageEnhance, ImageFilter
from collections import Counter
warnings.filterwarnings('ignore')

RUNNING_ON_KAGGLE = True  # True = Kaggle, False = Local

if RUNNING_ON_KAGGLE:
    _roots = [
        Path('/kaggle/input/the-2nd-ura-hackathon'),
        Path('/kaggle/input/competitions/the-2nd-ura-hackathon'),
    ]
    COMP_ROOT = next((p for p in _roots if p.exists()), _roots[0])

    # Tim thu muc anh test
    _img_cands = [
        COMP_ROOT / 'test_images' / 'test_images' / 'images',
        COMP_ROOT / 'test_images' / 'images',
        COMP_ROOT / 'images',
        COMP_ROOT / 'test' / 'images',
    ]
    IMAGES_DIR = next((p for p in _img_cands if p.is_dir()), _img_cands[0])

    TEST_CSV   = COMP_ROOT / 'test.csv'
    SAMPLE_CSV = COMP_ROOT / 'sample_submission.csv'
    TRAIN_CSV  = COMP_ROOT / 'train_labels.csv'
    WORK_DIR   = Path('/kaggle/working')
else:
    COMP_ROOT  = Path(r'C:\Users\LENOVO\Downloads\Urahackathon2026\the-2nd-ura-hackathon')
    IMAGES_DIR = COMP_ROOT / 'test_images' / 'images'
    TEST_CSV   = COMP_ROOT / 'test.csv'
    SAMPLE_CSV = COMP_ROOT / 'sample_submission.csv'
    TRAIN_CSV  = COMP_ROOT / 'train_labels.csv'
    WORK_DIR   = Path('.')

OUTPUT_CSV     = WORK_DIR / 'submission.csv'
CHECKPOINT_CSV = WORK_DIR / 'checkpoint.csv'

test_df = pd.read_csv(TEST_CSV)
train_labels_df = pd.read_csv(TRAIN_CSV, keep_default_na=False) if TRAIN_CSV.exists() else None

_all_imgs = list(IMAGES_DIR.glob('*')) if IMAGES_DIR.is_dir() else []
n_imgs = sum(1 for p in _all_imgs if p.suffix.lower() in ['.jpg','.jpeg','.png'])
print(f'COMP_ROOT  : {COMP_ROOT}  (exists={COMP_ROOT.exists()})')
print(f'IMAGES_DIR : {IMAGES_DIR}  (exists={IMAGES_DIR.is_dir()}, {n_imgs} imgs)')
if _all_imgs: print(f'  -> Vi du  : {_all_imgs[0].name}')
print(f'Test set   : {len(test_df):,} rows')
if train_labels_df is not None:
    has_prod = (train_labels_df['product_name'].str.strip()!='').sum()
    print(f'Train data : {len(train_labels_df):,} rows, co product: {has_prod:,}')
else:
    print('Train data : Khong tim thay!')


COMP_ROOT  : /kaggle/input/competitions/the-2nd-ura-hackathon  (exists=True)
IMAGES_DIR : /kaggle/input/competitions/the-2nd-ura-hackathon/test_images/test_images/images  (exists=True, 2006 imgs)
  -> Vi du  : img_6833.jpg
Test set   : 2,006 rows
Train data : 4,892 rows, co product: 2,900


In [5]:
IMAGE_PATH_INDEX = {}
for p in IMAGES_DIR.glob('*'):
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
        IMAGE_PATH_INDEX[p.stem] = str(p)
        IMAGE_PATH_INDEX[p.name] = str(p)

print(f'Indexed {len(IMAGE_PATH_INDEX):,} image paths')
print('Mẫu image_id:', test_df['image_id'].head(5).tolist())
_missing = [fid for fid in test_df['image_id'].head(5) if str(fid) not in IMAGE_PATH_INDEX]
print('Thiếu (nếu có):', _missing)

Indexed 4,012 image paths
Mẫu image_id: ['img_2934', 'img_2935', 'img_2936', 'img_2937', 'img_2938']
Thiếu (nếu có): []


In [3]:
import unicodedata

# ==================== UTILITY ====================

def _fold_ascii(s):
    s = str(s).replace('\u0111','d').replace('\u0110','D').casefold()
    s = unicodedata.normalize('NFD', s)
    return ''.join(c for c in s if unicodedata.category(c)!='Mn')

def strip_diacritics(text):
    return _fold_ascii(text)


# ==================== SOCIAL NOISE FILTER ====================
SOCIAL_NOISE_WORDS = frozenset({
    'tiktok','capcut','instagram','reels','facebook','youtube',
    'follow','like','share','comment','subscribe','livestream',
    'fyp','duet','stitch','viral','trending','video','clip','news',
})

OCR_TYPO_MAP = (
    (r'canfuc([o0])', r'canfoc\1'),
    (r'vinamill\b', 'vinamilk'),
    (r'vinamik\b',  'vinamilk'),
    (r'halong\b',   'ha long'),
    (r'patê',       'pate'),
)

def clean_social_ocr(text):
    import re
    text = re.sub(r'@[\w\.]+|#\w+|https?://\S+|www\.\S+', ' ', str(text))
    text = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', ' ', text)
    text = re.sub(r'(\d\s*){5,}', ' ', text)
    for pat, repl in OCR_TYPO_MAP:
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


PRODUCT_MIN_OCR_CHARS = 4

def _is_ocr_noise_only(text):
    text = str(text or '').strip()
    if len(text) < PRODUCT_MIN_OCR_CHARS: return True
    tokens = re.sub(
        r'[^\w\s\+\u00c0-\u1ef9\u0111\u0110]', ' ', text.lower()
    ).split()
    if not tokens: return True
    meaningful = [
        t for t in tokens
        if len(t)>=3 and t not in SOCIAL_NOISE_WORDS
        and not t.startswith('@') and 'www' not in t
    ]
    return len(meaningful)==0


# ==================== BRAND RULES v4.0 ====================
# Format: (regex, canonical_brand, [product_line_keywords])
BRAND_RULES = [
    # --- HALONG CANFOCO + PATE COT DEN ---
    (r'ha\s*long\s*canf[ou]c[o0].*pate.*c[\u1ed9o]t|c[\u1ed9o]t\s*[d\u0111][\u00e8e]n.*ha\s*long\s*canf[ou]c[o0]',
     'Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n', []),
    (r'ha\s*long\s*canf[ou]c[o0].*pate|canf[ou]c[o0].*pate\s*c[\u1ed9o]t|pate\s*c[\u1ed9o]t.*canf[ou]c[o0]',
     'Ha Long Canfoco Pate', []),
    (r'ha\s*long\s*canf[ou]c[o0]|halong\s*canf[ou]c[o0]|canfuc[o0]|canfood|\bcanf[ou]c[o0]\b',
     'Ha Long Canfoco', []),
    # THEM: 'canf' (OCR cat ngan)
    (r'halong\s*canf\b|ha\s*long\s*canf\b', 'Ha Long Canfoco', []),

    # --- DO HOP HA LONG ---
    (r'[d\u0111][\u1ed3o]\s*h[\u1ed9o]p\s*h[\u1ea1a]\s*long|do\s*hop\s*ha\s*long',
     '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long', []),
    # THEM: hophalong (OCR dich lien)
    (r'hophalong|hop\s*ha\s*long', '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long', []),

    # --- PATE COT DEN HAI PHONG ---
    (r'pate\s*c[\u1ed9o]t\s*[d\u0111][\u00e8e]n|pate\s*cot\s*den|c[\u1ed9o]t\s*[d\u0111][\u00e8e]n\s*h[\u1ea3a]i\s*ph[\u00f2o]ng',
     'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng', []),
    # THEM: cotcen, cotd variants
    (r'cotc[e]n.*hai|cot\s*c[e]n.*hai|cotd.*hai\s*ph[\u00f2o]ng',
     'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng', []),
    # THEM: bien the OCR [ate/fate/jate + cotcen/cotden
    (r'[\[fjh]?at[e\u00ea].*cot.*d[e\u00ea\u00e8n]|cotc[e]n|cot\s*c[e]n\b|\bcotd[e\u00ea]?\b',
     'Pate C\u1ed9t \u0110\u00e8n', []),

    # --- STANDARD BRANDS ---
    (r'pate\s*h[\u1ea1a]\s*long|h[\u1ea1a]\s*long\s*pate', 'Ha Long Canfoco Pate', []),
    (r'vinamilk|vinamill|vinamik', 'Vinamilk',
     ['flex','adm gold','sure','canxi','colosbaby','colos baby','ong tho','\u00f4ng th\u1ecd','dielac','grow']),
    (r'th\s*true|thtrue|th\s*true\s*milk', 'TH True Milk',
     ['true yogurt','grow','school milk','true butter']),
    (r'dutch\s*lady|c\u00f4\s*g\u00e1i|dutchlady', 'Dutch Lady',
     ['grow','complete','canxi','mom']),
    (r'nutifood|\bnuti\b', 'Nutifood', ['growplus','grow plus','pedia','iq']),
    (r'\bensure\b', 'Abbott Ensure', ['gold','original','plus']),
    (r'pediasure', 'Abbott PediaSure', []),
    (r'similac', 'Abbott Similac', []),
    (r'glucerna', 'Abbott Glucerna', []),
    (r'\bmilo\b', 'Nestl\u00e9 Milo', []),
    (r'nestle|nestl\u00e9', 'Nestl\u00e9',
     ['milo','coffee mate','carnation','nestea','nan','s\u1eefa b\u1ed9t']),
    (r'aptamil', 'Aptamil', ['first infant formula','profutura','gold']),
    (r'\bhipp\b', 'HiPP', ['combiotic','organic']),
    (r'\bfriso\b', 'Friso', ['gold','comfort','prestige']),
    (r'\bmeiji\b', 'Meiji', ['growing','step']),
    (r'nan\s*(optipro|opti\s*pro|infinipro|infini\s*pro|supremepro|supreme\s*pro|a2|ha)\b',
     'Nestl\u00e9 NAN', []),
    (r'sua\s*nan\b|nestle\s*nan|nestl\u00e9\s*nan', 'Nestl\u00e9 NAN', []),
    # THEM: dulac/dielac (OCR sai Dielac Vinamilk)
    (r'\bdulac\b|\bdielac\b', 'Vinamilk Dielac', []),
    (r'ba\s*vi\b|bavi\b|ba\s*v\u00ec', 'Ba V\u00ec', ['gold']),
    (r'lothamilk', 'Lothamilk', ['canxi']),
    (r'yomost', 'Yomost', []),
    (r'dalat\s*milk|\u0111\u00e0\s*l\u1ea1t', '\u0110\u00e0 L\u1ea1t Milk', []),
    (r'\bkun\b|kun\s*milk', 'Kun', ['chocolate','strawberry']),
    (r'\bfami\b', 'Fami', ['canxi','kid']),
    (r'anlene', 'Anlene', ['gold','concentrate']),
    (r'\banchor\b', 'Anchor', ['butter','cream']),
    (r'vissan', 'Vissan',
     ['pate heo','pate ga','pate g\u00e0','xuc xich','x\u00fac x\u00edch','lap xuong','l\u1ea1p x\u01b0\u1edfng']),
    (r'\bhafi\b', 'Hafi', ['pate']),
    (r'ba\s*huan|ba\s*hu\u00e2n', 'Ba Hu\u00e2n', ['pate']),
    (r'san\s*ha\b|san\s*h\u00e0', 'San H\u00e0', ['pate']),
    (r'\bcp\b|c\.p\.', 'CP', ['pate','x\u00fac x\u00edch']),
    (r'long\s*bien|long\s*bi\u00ean', 'Long Bi\u00ean', ['pate']),
    (r'highlands?\s*coffee', 'Highlands Coffee',
     ['tra sen vang','tra vai','banh mi que','americano']),
    (r'the\s*coffee\s*house', 'The Coffee House', ['tra phuc kien']),
    (r'nhan\s*hoa\s*foods?', 'Nh\u00e2n H\u00f2a Foods', ['pate']),
    (r'\bacnes\b', 'Acnes', ['vitamin cleanser']),
    (r'\bpate\b|pat\u00ea', 'Pate', []),
]

_SUB_NAMES = {
    'flex':'Flex','adm gold':'ADM Gold','sure':'Sure','canxi':'Canxi',
    'colosbaby':'ColosBaby','colos baby':'Colos Baby',
    'ong tho':'\u00d4ng Th\u1ecd','\u00f4ng th\u1ecd':'\u00d4ng Th\u1ecd',
    'dielac':'Dielac','grow':'Grow','true yogurt':'True Yogurt',
    'school milk':'School Milk','true butter':'True Butter',
    'growplus':'GrowPlus+','grow plus':'GrowPlus+','pedia':'Pedia','iq':'IQ',
    'gold':'Gold','original':'Original','plus':'Plus',
    'milo':'Milo','coffee mate':'Coffee Mate','nan':'NAN','s\u1eefa b\u1ed9t':'S\u1eefa B\u1ed9t',
    'first infant formula':'First Infant Formula','profutura':'Profutura',
    'combiotic':'Combiotic','organic':'Organic',
    'growing':'Growing','step':'Step',
    'comfort':'Comfort','prestige':'Prestige',
    'pate heo':'Pate Heo','pate ga':'Pate G\u00e0','pate g\u00e0':'Pate G\u00e0',
    'xuc xich':'X\u00fac X\u00edch','x\u00fac x\u00edch':'X\u00fac X\u00edch',
    'lap xuong':'L\u1ea1p X\u01b0\u1edfng','l\u1ea1p x\u01b0\u1edfng':'L\u1ea1p X\u01b0\u1edfng',
    'pate':'Pate','x\u00fac x\u00edch':'X\u00fac X\u00edch',
    'tra sen vang':'Tr\u00e0 Sen V\u00e0ng','tra vai':'Tr\u00e0 V\u1ea3i',
    'banh mi que':'B\u00e1nh M\u00ec Que','americano':'Americano',
    'tra phuc kien':'Tr\u00e0 Ph\u00fac Ki\u1ebfn',
    'vitamin cleanser':'Vitamin Cleanser','concentrate':'Concentrate',
    'butter':'Butter','cream':'Cream','chocolate':'Chocolate','strawberry':'Strawberry',
    'kid':'Kid','kid+':'Kid+',
}


def _normalize_for_rules(text):
    tl = text.lower().replace('pat\u00ea','pate')
    tl = re.sub(r'vina[jilm1]{0,1}milk', 'vinamilk', tl, flags=re.IGNORECASE)
    tl = re.sub(r'canfuc([o0])', r'canfoc\1', tl, flags=re.IGNORECASE)
    tl = re.sub(r'halong\b', 'ha long', tl, flags=re.IGNORECASE)
    return tl


# --- TANAHIVER6: anti-hallucination token filter ---
def _build_brand_vocab():
    vocab = set()
    for _pat, brand, lines in BRAND_RULES:
        for w in brand.split(): vocab.add(w.casefold())
        for line in lines:
            for w in line.title().split(): vocab.add(w.casefold())
    return frozenset(vocab)

_BRAND_VOCAB = _build_brand_vocab()
_BRAND_VOCAB_FOLDED = frozenset(_fold_ascii(t) for t in _BRAND_VOCAB)


def _ocr_word_set(text):
    tl = _normalize_for_rules(text)
    return {_fold_ascii(w) for w in re.findall(r'[^\W\d_]+', tl, flags=re.UNICODE) if w}


def _token_in_ocr(tok, ocr_text, ocr_words):
    tf = _fold_ascii(tok)
    if tf in ocr_words: return True
    compact = _fold_ascii(ocr_text).replace(' ','')
    if len(tf)>=2 and tf in compact: return True
    return any(len(tf)>=2 and tf in ow for ow in ocr_words)


def _assign_readable_brand_tokens(ocr_text, candidate):
    '''Chi giu token thuoc brand list VA doc duoc tu ocr_text (chong hallucination)'''
    if not candidate or not candidate.strip(): return ''
    ocr_words = _ocr_word_set(ocr_text)
    kept = []
    for tok in candidate.split():
        if _fold_ascii(tok) not in _BRAND_VOCAB_FOLDED: continue
        if _token_in_ocr(tok, ocr_text, ocr_words): kept.append(tok)
    return ' '.join(kept)


def extract_by_rules(text):
    if not text: return ''
    tl = _normalize_for_rules(text)
    matched = ''
    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE):
            for line in lines:
                if re.search(line, tl, re.IGNORECASE):
                    sub = _SUB_NAMES.get(line, line.title())
                    matched = f'{brand} {sub}'.strip()
                    break
            if not matched:
                matched = brand
            break
    return _assign_readable_brand_tokens(text, matched)


def post_process_prediction(pred, ocr_text):
    '''Upgrade 'Pate' generic -> cu the hon bang cach re-scan OCR'''
    if not pred: return pred
    nd = _fold_ascii(ocr_text) if ocr_text else ''
    if pred == 'Pate':
        if re.search(r'hai\s*ph[o]ng', nd):
            if re.search(r'cot|cotcen|cotd|col\s*d', nd):
                return 'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng'
        if any(re.search(p, nd) for p in [r'cot\s*d[e]n',r'\bcotd[e]?\b',r'cotcen',r'cot\s*cen']):
            return 'Pate C\u1ed9t \u0110\u00e8n'
        if re.search(r'ha\s*long|halong', nd):
            return '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long'
    if pred == 'Nestl\u00e9':
        if re.search(r'\bnan\b|nan\s*opti|nan\s*infini|nan\s*supreme', nd):
            return 'Nestl\u00e9 NAN'
        if re.search(r'\bmilo\b', nd):
            return 'Nestl\u00e9 Milo'
    if pred == 'Vinamilk':
        if re.search(r'dulac|dielac', nd):
            return 'Vinamilk Dielac'
    return pred
# ==================== PRODUCT NAME NORMALIZATION ====================
_PRODUCT_CANON_MAP = None

def _build_product_canon_map():
    """Gop cac bien the chinh ta/dau cau cua cung 1 san pham (VD: 'côt' vs 'cột',
    'patê' vs 'pate') bang fold-key, chon spelling pho bien nhat trong train data
    lam chinh tac. Manh hon Title Case don thuan vi xu ly duoc ca loi dau cau."""
    global _PRODUCT_CANON_MAP
    if _PRODUCT_CANON_MAP is not None:
        return _PRODUCT_CANON_MAP
    _PRODUCT_CANON_MAP = {}
    try:
        names = train_labels_df['product_name'].astype(str).str.strip()
        names = names[names != '']
        counts = Counter(unicodedata.normalize('NFC', n) for n in names)
        groups = {}
        for name, cnt in counts.items():
            key = _fold_ascii(name)
            groups.setdefault(key, []).append((cnt, name))
        for key, variants in groups.items():
            # uu tien: nhieu mau nhat -> neu hoa, chon ten day du dau nhat
            variants.sort(key=lambda x: (-x[0], -len(x[1])))
            _, best_name = variants[0]
            _PRODUCT_CANON_MAP[key] = best_name.title()
    except Exception:
        pass
    return _PRODUCT_CANON_MAP


def normalize_product_name(name):
    """Chuan hoa nhan san pham: NFC + khoang trang + gop bien the
    viet hoa/thuong VA chinh ta/dau cau ve 1 ten chinh tac duy nhat."""
    if not name:
        return ''
    name = unicodedata.normalize('NFC', str(name).strip())
    name = re.sub(r'\s+', ' ', name)
    if not name:
        return ''
    canon_map = _build_product_canon_map()
    key = _fold_ascii(name)
    return canon_map.get(key, name.title())

# Smoke test
_tests = [
    ('HA LONG CANFOCO Pate C\u1ed9t \u0110\u00e8n', 'Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n'),
    ('HA LONG CANFOCO pate cot den Hai Phong',    'Ha Long Canfoco Pate'),
    ('HALONG CANFUCO j TIkTok',                   'Ha Long Canfoco'),
    ('\u0110\u1ed3 h\u1ed9p H\u1ea1 Long ISO22000', '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long'),
    ('Pate c\u1ed9t \u0111\u00e8n',               'Pate C\u1ed9t \u0110\u00e8n'),
    ('Vinamilk Flex 180ml',                       'Vinamilk Flex'),
    ('MILO Nestle chocolate 3 in 1',              'Nestl\u00e9 Milo'),
    ('Pate Heo Vissan 170g',                      'Vissan Pate Heo'),
    ('TH True Milk tuoi tiet trung',              'TH True Milk'),
    ('Dutch Lady Grow+ 900g',                     'Dutch Lady Grow'),
    ('No brand text',                             ''),
    ('tiktok capcut viral',                       ''),
    ('Pate COTDEN 130 tan thit lon',              'Pate'),  # cotden in text but maybe
]
print('Rules v4.0 | %d rules | Smoke test:' % len(BRAND_RULES))
ok_n=0
for txt, exp in _tests:
    r = extract_by_rules(txt)
    ok = (exp=='' and r=='') or (exp!='' and exp.split()[0].lower() in r.lower())
    if ok: ok_n+=1
    print('  [%s] %-45s => %s' % ('OK' if ok else '??', txt[:45], repr(r)[:40]))
print('  Score: %d/%d' % (ok_n, len(_tests)))


Rules v4.0 | 46 rules | Smoke test:
  [OK] HA LONG CANFOCO Pate Cột Đèn                  => 'Ha Long Canfoco Pate Cột Đèn'
  [OK] HA LONG CANFOCO pate cot den Hai Phong        => 'Ha Long Canfoco Pate Cột Đèn'
  [OK] HALONG CANFUCO j TIkTok                       => 'Ha Long Canfoco'
  [OK] Đồ hộp Hạ Long ISO22000                       => 'Đồ Hộp Hạ Long'
  [OK] Pate cột đèn                                  => 'Pate Cột Đèn'
  [OK] Vinamilk Flex 180ml                           => 'Vinamilk Flex'
  [OK] MILO Nestle chocolate 3 in 1                  => 'Nestlé Milo'
  [OK] Pate Heo Vissan 170g                          => 'Vissan Pate Heo'
  [OK] TH True Milk tuoi tiet trung                  => 'TH True Milk'
  [OK] Dutch Lady Grow+ 900g                         => 'Dutch Lady Grow'
  [OK] No brand text                                 => ''
  [OK] tiktok capcut viral                           => ''
  [OK] Pate COTDEN 130 tan thit lon                  => 'Pate Cột Đèn'
  Score: 13/13


In [6]:
import os, threading, numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

OCR_CONF_THRESHOLD = 0.35
_ocr_worker_local = threading.local()
NUM_WORKERS = max(1, min(CPU_THREADS, 4))


def _latin_dict_path():
    import paddleocr
    return str(
        Path(paddleocr.__file__).resolve().parent
        / 'ppocr' / 'utils' / 'dict' / 'latin_dict.txt'
    )


def _paddle_model_dirs():
    whl = Path.home() / '.paddleocr' / 'whl'
    return (
        whl / 'det' / 'ch' / 'ch_PP-OCRv4_det_infer',
        whl / 'rec' / 'latin' / 'latin_PP-OCRv3_rec_infer',
    )


def _paddle_model_ready(d):
    d = Path(d)
    return (d/'inference.pdmodel').is_file() and (d/'inference.pdiparams').is_file()


_PADDLE_URLS = (
    'https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_infer.tar',
    'https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/latin_PP-OCRv3_rec_infer.tar',
)


def _prefetch_models():
    from ppocr.utils.network import maybe_download
    det_dir, rec_dir = _paddle_model_dirs()
    pending = [
        (str(det_dir), _PADDLE_URLS[0]),
        (str(rec_dir), _PADDLE_URLS[1]),
    ]
    pending = [(d,u) for d,u in pending if not _paddle_model_ready(d)]
    if not pending: return det_dir, rec_dir
    workers = min(NUM_WORKERS, 2)
    print(f'Prefetch {len(pending)} PaddleOCR model(s)...')
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futs = {pool.submit(maybe_download,d,u): Path(d).name for d,u in pending}
        for fut in as_completed(futs):
            fut.result(); print(f'  ready: {futs[fut]}')
    return det_dir, rec_dir


def _make_paddle_ocr():
    from paddleocr import PaddleOCR
    det_dir, rec_dir = _prefetch_models()
    return PaddleOCR(
        use_gpu=False,
        use_angle_cls=True,
        show_log=False,
        enable_mkldnn=True,
        lang='latin',
        det_model_dir=str(det_dir),
        rec_model_dir=str(rec_dir),
        rec_char_dict_path=_latin_dict_path(),
    )


def preprocess(img, max_dim=1280):
    w, h = img.size
    if max(w,h) > max_dim:
        r = max_dim/max(w,h)
        img = img.resize((int(w*r), int(h*r)), Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(1.35)
    return img.filter(ImageFilter.SHARPEN)


def postprocess_ocr(text):
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    if not tokens: return ''
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower(): deduped.append(tok)
    return ' '.join(deduped)


def _paddle_ocr_text(engine, img_np_bgr):
    raw = engine.ocr(img_np_bgr, cls=True)
    if not raw or not raw[0]: return ''
    items = []
    for box, (text, conf) in raw[0]:
        if conf > OCR_CONF_THRESHOLD:
            yx = box[0]
            items.append((yx[1], yx[0], text))
    items.sort()
    return postprocess_ocr(' '.join(t for _,_,t in items))


def load_image(image_id):
    path = IMAGE_PATH_INDEX.get(str(image_id))
    if path is None or not Path(path).exists(): return None
    try:
        return Image.open(str(path)).convert('RGB')
    except Exception: return None


def run_paddle_ocr(image_path):
    try:
        img = Image.open(str(image_path)).convert('RGB')
    except Exception: return ''
    try:
        img = preprocess(img)
        img_np = np.array(img)[:,:,::-1]
        return _paddle_ocr_text(ocr_engine, img_np)
    except Exception: return ''

_engine_create_lock = threading.Lock()
def _ocr_image_only(image_id, _images_dir_str):
    engine = getattr(_ocr_worker_local, 'engine', None)
    if engine is None:
        os.environ['OMP_NUM_THREADS'] = '1'
        os.environ['MKL_NUM_THREADS'] = '1'
        _ocr_worker_local.engine = _make_paddle_ocr()
    img = load_image(image_id)
    if img is None: return image_id, ''
    img = preprocess(img)
    img_np = np.array(img)[:,:,::-1]
    return image_id, _paddle_ocr_text(_ocr_worker_local.engine, img_np)


print('Loading PaddleOCR (CPU, latin_PP-OCRv3)...')
ocr_engine = _make_paddle_ocr()
print(f'PaddleOCR OK | conf>{OCR_CONF_THRESHOLD} | {NUM_WORKERS} parallel worker(s)')

print('\nSmoke test on first image...')
_fid = test_df['image_id'].iloc[0]
_img = load_image(_fid)
if _img:
    _arr = np.array(preprocess(_img))[:,:,::-1]
    _ocr = _paddle_ocr_text(ocr_engine, _arr)
    print(f'  image_id : {_fid}')
    print(f'  ocr_text : {_ocr[:80]}{"..." if len(_ocr)>80 else ""}')
else:
    print(f'  Warning: {_fid} not found')


Loading PaddleOCR (CPU, latin_PP-OCRv3)...
PaddleOCR OK | conf>0.35 | 4 parallel worker(s)

Smoke test on first image...
  image_id : img_2934
  ocr_text : TFEDTIENTHAING-UTCNEWS ANHRTENCHIP HALONGECANFOCO SO22000:2018FSSC22000 vate d T...


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SkPipeline


class ProductPredictor:
    def __init__(self, min_class=3, prob_thr=0.60, max_feat=3000):
        self.min_class=min_class; self.prob_thr=prob_thr; self.max_feat=max_feat
        self._has_clf=self._prod_clf=None; self.fitted=False

    def fit(self, df, rule_fn):
        df=df.copy()
        df['ocr_text']=df['ocr_text'].astype(str).str.strip()
        df['product_name']=df['product_name'].astype(str).str.strip()
        self._rule_fn=rule_fn
        # Binary: has product?
        self._has_clf=SkPipeline([
            ('tfidf', TfidfVectorizer(analyzer='char_wb',ngram_range=(2,4),max_features=self.max_feat,min_df=2)),
            ('clf',   LogisticRegression(max_iter=400, class_weight='balanced')),
        ])
        self._has_clf.fit(df['ocr_text'], (df['product_name']!='').astype(int))
        # Multi-class: which product?
        pos=df[(df['ocr_text']!='')&(df['product_name']!='')]
        keep=pos['product_name'].value_counts()
        keep=keep[keep>=self.min_class].index
        pos=pos[pos['product_name'].isin(keep)]
        self._prod_clf=SkPipeline([
            ('tfidf', TfidfVectorizer(analyzer='char_wb',ngram_range=(2,4),max_features=self.max_feat,min_df=2)),
            ('clf',   LogisticRegression(max_iter=400, class_weight='balanced')),
        ])
        if len(pos):
            self._prod_clf.fit(pos['ocr_text'], pos['product_name'])
        self.fitted=True
        print(f'  ProductPredictor: {len(df):,} rows, {pos["product_name"].nunique() if len(pos) else 0} classes')
        return self

    def predict(self, ocr_text):
        ocr_text=str(ocr_text or '').strip()
        if not ocr_text or _is_ocr_noise_only(ocr_text): return ''
        ruled=self._rule_fn(ocr_text)
        if ruled: return post_process_prediction(ruled, ocr_text)
        if not self.fitted or self._has_clf is None: return ''
        proba=self._has_clf.predict_proba([ocr_text])[0]
        classes=list(self._has_clf.classes_)
        if 1 not in classes or proba[classes.index(1)] < self.prob_thr: return ''
        pred=str(self._prod_clf.predict([ocr_text])[0])
        result=_assign_readable_brand_tokens(ocr_text, pred)
        return post_process_prediction(result, ocr_text) if result else ''


product_predictor=None
if train_labels_df is not None:
    print('Training ProductPredictor...')
    product_predictor=ProductPredictor(min_class=3, prob_thr=0.60, max_feat=3000)
    product_predictor.fit(train_labels_df, extract_by_rules)
    print('ProductPredictor ready.')
else:
    print('train_labels.csv not found - rules-only mode')

print('Extraction v4.0 OK')


Training ProductPredictor...
  ProductPredictor: 4,892 rows, 249 classes
ProductPredictor ready.
Extraction v4.0 OK


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

class SklearnPredictor:
    def __init__(self,min_cls=2,thr=0.50,mf=6000):
        self.min_cls=min_cls; self.thr=thr; self.mf=mf
        self._gate=self._prod=None; self.fitted=False
    def _pipe(self):
        return Pipeline([('tf',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,4),max_features=self.mf,sublinear_tf=True)),
                         ('clf',LogisticRegression(max_iter=800,class_weight='balanced',C=1.5,solver='lbfgs'))])
    def fit(self,df):
        df=df.copy()
        df['ocr_text']=df['ocr_text'].astype(str).str.strip()
        df['product_name']=df['product_name'].astype(str).str.strip().apply(normalize_product_name)
        df=df[df['ocr_text']!='']
        yg=(df['product_name']!='').astype(int)
        self._gate=self._pipe().fit(df['ocr_text'],yg)
        pos=df[df['product_name']!='']
        vc=pos['product_name'].value_counts()
        pos=pos[pos['product_name'].isin(vc[vc>=self.min_cls].index)]
        if len(pos)>10: self._prod=self._pipe().fit(pos['ocr_text'],pos['product_name'])
        self.fitted=True
        print('  Sklearn: %d pos, %d classes.' % (yg.sum(), len(vc[vc>=self.min_cls])))
        return self
    def predict(self,text):
        text=str(text).strip() if text else ''
        if not text or not self.fitted or self._gate is None: return ''
        try:
            gp=self._gate.predict_proba([text])[0]
            cl=list(self._gate.classes_)
            if 1 not in cl or gp[cl.index(1)]<self.thr: return ''
            if self._prod is None: return ''
            return normalize_product_name(str(self._prod.predict([text])[0]))
        except: return ''

class KNNPredictor:
    def __init__(self,k=5,thr=0.60):
        self.k=k; self.thr=thr
        self._tf=self._mat=None; self._labels=[]; self.fitted=False
    def fit(self,df):
        df=df.copy()
        df['ocr_text']=df['ocr_text'].astype(str).str.strip()
        df['product_name']=df['product_name'].astype(str).str.strip().apply(normalize_product_name)
        df=df[df['ocr_text']!='']
        self._tf=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,4),max_features=6000,sublinear_tf=True)
        self._mat=self._tf.fit_transform(df['ocr_text'])
        self._labels=df['product_name'].tolist()
        self.fitted=True
        print('  KNN: %d vectors.' % len(self._labels))
        return self
    def predict(self,text):
        if not self.fitted or not text or len(text.strip())<5: return ''
        try:
            sv=self._tf.transform([text])
            sims=cosine_similarity(sv,self._mat).flatten()
            idx=sims.argsort()[::-1][:self.k]
            votes=[self._labels[i] for i in idx if sims[i]>=self.thr and self._labels[i]]
            if votes: return Counter(votes).most_common(1)[0][0]
        except: pass
        return ''

sklearn_pred = knn_pred = None
if train_labels_df is not None:
    print('Training Sklearn (L4)...')
    sklearn_pred = SklearnPredictor(min_cls=2,thr=0.50,mf=6000).fit(train_labels_df)
    print('Training KNN (L5)...')
    knn_pred = KNNPredictor(k=5,thr=0.60).fit(train_labels_df)
else:
    print('No train data -> skip L4/L5.')


Training Sklearn (L4)...
  Sklearn: 2876 pos, 228 classes.
Training KNN (L5)...
  KNN: 3973 vectors.


In [9]:
def predict_product(ocr_text, image_id=None):
    ocr_text=str(ocr_text or '').strip()
    if _is_ocr_noise_only(ocr_text): return ''
    if product_predictor is not None:
        return product_predictor.predict(ocr_text)
    # fallback rules-only
    r=extract_by_rules(ocr_text)
    return post_process_prediction(r, ocr_text) if r else ''


_P=[
    ('\u0110\u1ed3 h\u1ed9p H\u1ea1 Long', '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long'),
    ('HALONG CANFOCO Pate C\u1ed9t \u0110\u00e8n','Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n'),
    ('ate Jate Cotcen 130 tan thit lon', 'Pate C\u1ed9t \u0110\u00e8n'),
    ('HIGHLANDS COFFEE tra sen vang', 'Highlands Coffee Tr\u00e0 Sen V\u00e0ng'),
    ('tiktok viral capcut fyp news', ''),
    ('HiPP Combiotic organic milk', 'HiPP Combiotic'),
    ('Nestlé NAN OPTIpro 0-6', 'Nestl\u00e9 NAN'),
    ('hophalong thu hoi san pham', '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long'),
]
print('Pipeline v4.0 test:')
ok_n=0
for txt, exp in _P:
    r=predict_product(txt)
    ok=(exp=='' and r=='') or (exp!='' and exp.split()[0].lower() in r.lower())
    if ok: ok_n+=1
    print('  [%s] %-45s => %s' % ('OK' if ok else 'FAIL', txt[:45], repr(r)[:45]))
print('  Score: %d/%d' % (ok_n, len(_P)))


Pipeline v4.0 test:
  [OK] Đồ hộp Hạ Long                                => 'Đồ Hộp Hạ Long'
  [OK] HALONG CANFOCO Pate Cột Đèn                   => 'Ha Long Canfoco Pate Cột Đèn'
  [FAIL] ate Jate Cotcen 130 tan thit lon              => 'Cột'
  [OK] HIGHLANDS COFFEE tra sen vang                 => 'Highlands Coffee Trà Sen Vàng'
  [OK] tiktok viral capcut fyp news                  => ''
  [OK] HiPP Combiotic organic milk                   => 'HiPP Combiotic'
  [OK] Nestlé NAN OPTIpro 0-6                        => 'Nestlé NAN'
  [FAIL] hophalong thu hoi san pham                    => 'Hộp Hạ Long'
  Score: 6/8


In [10]:
def evaluate_pipeline(df_val, n=500):
    if len(df_val)>n: df_val=df_val.sample(n,random_state=42)
    tp=fp=fn=exact=ev=0
    for _,row in tqdm(df_val.iterrows(),total=len(df_val),desc='Eval'):
        ocr=str(row['ocr_text']).strip()
        gt=normalize_product_name(str(row['product_name']).strip())
        if not ocr: continue
        pred=normalize_product_name(predict_product(ocr))
        pt=set(pred.lower().split()) if pred else set()
        gt_=set(gt.lower().split()) if gt else set()
        tp+=len(pt&gt_); fp+=len(pt-gt_); fn+=len(gt_-pt)
        if pred.lower()==gt.lower(): exact+=1
        ev+=1
    p=tp/(tp+fp) if tp+fp>0 else 0
    r=tp/(tp+fn) if tp+fn>0 else 0
    f1=2*p*r/(p+r) if p+r>0 else 0
    return {'F1':f1,'P':p,'R':r,'Exact':exact/ev if ev>0 else 0,'N':ev}

if train_labels_df is not None:
    _v=train_labels_df.sample(frac=0.2,random_state=999)
    _v=_v[_v['ocr_text'].astype(str).str.strip()!='']
    print('Eval %d samples...' % min(500,len(_v)))
    m=evaluate_pipeline(_v,500)
    print('F1=%.4f P=%.4f R=%.4f ExactAcc=%.4f N=%d' % (m['F1'],m['P'],m['R'],m['Exact'],m['N']))
    print('Est.Score (CER~0.25): %.4f' % (0.6*m['F1']+0.4*0.75))
else:
    print('No train data.')


Eval 500 samples...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]

F1=0.5549 P=0.6413 R=0.4890 ExactAcc=0.4560 N=500
Est.Score (CER~0.25): 0.6329


In [11]:
from joblib import Parallel, delayed

SAVE_EVERY = 50
CHECKPOINT_CSV = WORK_DIR / 'checkpoint.csv'

# Resume tu checkpoint
results, done_ids = [], set()
if CHECKPOINT_CSV.exists():
    ckpt = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    done_ids = set(ckpt['image_id'])
    results = ckpt.to_dict('records')
    print(f'Resume: {len(done_ids):,} done.')
else:
    print('Fresh start.')

pending = [i for i in test_df['image_id'] if i not in done_ids]
print(f'Pending: {len(pending):,} | Workers: {NUM_WORKERS}')

images_dir_str = str(IMAGES_DIR)

import time
errors = 0
t0 = time.perf_counter()

for batch_start in tqdm(range(0, len(pending), SAVE_EVERY), desc='OCR batches'):
    batch = pending[batch_start: batch_start+SAVE_EVERY]

    # Parallel OCR
    ocr_pairs = Parallel(n_jobs=NUM_WORKERS, backend='threading', batch_size=4)(
        delayed(_ocr_image_only)(iid, images_dir_str) for iid in batch
    )

    # Product prediction (main process)
    for image_id, ocr_text in ocr_pairs:
        try:
            cleaned = clean_social_ocr(ocr_text)
            product = predict_product(cleaned, image_id)
            results.append({
                'image_id': image_id,
                'ocr_text': ocr_text,
                'product_name': product,
            })
        except Exception as e:
            results.append({'image_id': image_id, 'ocr_text': '', 'product_name':''})
            errors += 1

    # Save checkpoint
    pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8')

elapsed = time.perf_counter() - t0
print(f'\nDone {len(results)} imgs in {elapsed/60:.1f} min ({elapsed/max(len(results),1):.2f}s/img)')
n_ocr = sum(1 for r in results if str(r.get('ocr_text','')).strip())
n_prod = sum(1 for r in results if str(r.get('product_name','')).strip())
print(f'OCR ok: {n_ocr} | Product: {n_prod} | Errors: {errors}')


Fresh start.
Pending: 2,006 | Workers: 4


OCR batches:   0%|          | 0/41 [00:00<?, ?it/s]


Done 2006 imgs in 20.8 min (0.62s/img)
OCR ok: 1700 | Product: 1029 | Errors: 0


In [12]:
import csv as _csv

sub = pd.DataFrame(results)[['image_id','ocr_text','product_name']].copy()
for col in ('ocr_text','product_name'):
    sub[col] = sub[col].fillna('').astype(str).str.strip()
    sub.loc[sub[col]=='', col] = ' '

sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
checks = {
    'AC1 rows': len(sub)==len(sample),
    'AC2 no extra': not bool(set(sub['image_id'])-set(sample['image_id'])),
    'AC3 no miss': not bool(set(sample['image_id'])-set(sub['image_id'])),
    'AC4 no dup': not sub['image_id'].duplicated().any(),
    'AC5 no null': not sub.isnull().any().any(),
    'AC6 no newline': not sub['ocr_text'].str.contains(r'[\n\t]',regex=True,na=False).any(),
    'AC7 col order': list(sub.columns)==['image_id','ocr_text','product_name'],
}
ok = all(checks.values())
for k,v in checks.items(): print('  [%s] %s' % ('PASS' if v else 'FAIL', k))

if ok:
    sub = sub.set_index('image_id').reindex(sample['image_id']).reset_index()
    sub.to_csv(OUTPUT_CSV, index=False, encoding='utf-8', quoting=_csv.QUOTE_ALL)
    sc = sub.copy(); sc['product_name'] = sc['product_name'].str.strip()
    ne = (sc['product_name']=='').sum()
    top = sc[sc['product_name']!='']['product_name'].value_counts().head(10)
    print('Saved: %s (%.1f KB)' % (OUTPUT_CSV, OUTPUT_CSV.stat().st_size/1024))
    print('No product: %d (%.1f%%)' % (ne, ne/len(sub)*100))
    print('Top 10:')
    for nm,cnt in top.items(): print('  %5dx %s' % (cnt, nm))
else:
    print('[ERROR] Some ACs failed!')


  [PASS] AC1 rows
  [PASS] AC2 no extra
  [PASS] AC3 no miss
  [PASS] AC4 no dup
  [PASS] AC5 no null
  [PASS] AC6 no newline
  [PASS] AC7 col order
Saved: /kaggle/working/submission.csv (238.2 KB)
No product: 977 (48.7%)
Top 10:
    276x Đồ Hộp Hạ Long
    159x Ha Long Canfoco
     91x Pate Cột Đèn Hải Phòng
     69x Pate Cột Đèn
     52x Nestlé
     40x Highlands Coffee
     36x Pate
     30x Nestlé NAN
     27x Aptamil
     26x CP
